In [1]:
import os
import uuid
from langchain.text_splitter import RecursiveCharacterTextSplitter

sub_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

In [ ]:
import os
import json
import re
import time
import requests

API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={API_KEY}"

headers = {
    "Content-Type": "application/json"
}

input_dir = "/kaggle/input/his-chunk-new-16-10/his_chunks"
output_path = "/kaggle/working/history_triples_v3.json"
processed_keys = set()

# === BƯỚC 2: Nếu file kết quả đã có, đọc các đoạn đã xử lý ===
if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line.strip())
                key = (record["formId"], record["chunk_index"])
                processed_keys.add(key)
            except:
                continue

print(f"Đã tải {len(processed_keys)} đoạn đã xử lý trước đó.")

json_files = sorted([f for f in os.listdir(input_dir) if f.endswith(".json")])
print(f"Tổng số file JSON cần xử lý: {len(json_files)}")

# # === BƯỚC 3: Đọc file JSON input ===
# with open("/kaggle/input/his-chunk-new-16-10/his_chunks/LS1.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# === BƯỚC 5: Gửi request API từng đoạn và lưu từng dòng ===
MAX_RETRIES = 5
BASE_DELAY = 2  # giây
WAIT_AFTER_N_REQUESTS = 10
LONG_SLEEP_SECONDS = 60

request_count = 0  # Đếm số request thành công liên tiếp

with open(output_path, "a", encoding="utf-8") as out_f:
    for filename in json_files:
        file_path = os.path.join(input_dir, filename)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"Lỗi đọc {filename}: {e}")
            continue

        print(f"\n=== Xử lý file: {filename} ({len(data)} chunk lớn) ===")
        for form in data:
            form_id = form.get("chunk_id", "unknown")
            content = form.get("text", "").strip()
    
            if not content:
                continue
    
            chunks = sub_splitter.split_text(content)
    
            for i, chunk in enumerate(chunks):
                key = (form_id, i)
                if key in processed_keys:
                    print(f"Bỏ qua {form_id} đoạn {i} (đã xử lý)")
                    continue
    
                prompt = f"""Bạn là một công cụ trích xuất thông tin để xây dựng đồ thị tri thức du lịch – lịch sử Việt Nam.
                Mục tiêu: Từ một đoạn văn bản đầu vào (tiếng Việt), hãy trích xuất các bộ ba (triple) theo định dạng JSON:[{{"subject": "...", "subject_type": "...", "relation": "...", "object": "...", "object_type": "..."}}]
                - hãy suy luận hợp lý từ văn cảnh.
                Yêu cầu:
                - Mỗi dòng đại diện cho một quan hệ.
                - giữa các chữ nối nhau bằng dấu "_".
                - Không nên dùng danh từ làm tên relation.
                - Không nên dùng quan hệ quá dài hoặc mang nghĩa mơ hồ như "liên_quan_đến", "có_liên_hệ_với" → gây loãng và khó truy vấn.
    
                Đoạn văn đầu vào:
                {chunk}
                → Triple:
                Hãy trả về chỉ các triple dưới dạng JSON như trên, không thêm lời giải thích."""
    
                payload = {
                    "contents": [
                        {
                            "parts": [
                                {"text": prompt.strip()}
                            ]
                        }
                    ]
                }
    
                for attempt in range(1, MAX_RETRIES + 1):
                    try:
                        response = requests.post(API_URL, headers=headers, json=payload)
                        if response.status_code == 429:
                            raise Exception("429 Too Many Requests")
    
                        response.raise_for_status()
                        res_data = response.json()
                        generated_text = res_data["candidates"][0]["content"]["parts"][0]["text"]
    
                        # Bỏ markdown wrapper nếu có
                        if generated_text.startswith("```json"):
                            generated_text = re.sub(r"^```json\s*", "", generated_text)
                        if generated_text.endswith("```"):
                            generated_text = re.sub(r"\s*```$", "", generated_text)
    
                        print(f"Đã xử lý: {form_id} - đoạn {i+1}/{len(chunks)}")
    
                        # match = re.search(r"```json\s*(.*?)\s*```", generated_text, re.DOTALL)
                        # if match:
                        #     json_text = match.group(1).strip()
                        # else:
                        #     json_text = generated_text.strip()  # fallback nếu không có markdown wrapper
    
                        try:
                            parsed = json.loads(generated_text.strip())
                            # parsed = json.loads(json_text)
                            if isinstance(parsed, list):  # Nếu trả về list
                                triples = parsed
                            elif isinstance(parsed, dict) and "triples" in parsed:
                                triples = parsed["triples"]
                            else:
                                raise ValueError("Không nhận được format list hoặc dict hợp lệ.")
                        except Exception as parse_err:
                            print(f"Không parse được JSON: {parse_err}")
                            print("Nội dung trả về:\n", response.text)
                            triples = []
    
                        record = {
                            "formId": form_id,
                            "chunk_index": i,
                            "triples": triples,
                            "text": chunk,
                            "file": filename
                        }
    
                        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        out_f.flush()
    
                        request_count += 1
                        if request_count % WAIT_AFTER_N_REQUESTS == 0:
                            print(f"Nghỉ {LONG_SLEEP_SECONDS} giây sau {WAIT_AFTER_N_REQUESTS} request...")
                            time.sleep(LONG_SLEEP_SECONDS)
                        else:
                            time.sleep(2)  # delay ngắn giữa các request
    
                        break  # thành công thì thoát retry loop
    
                    except Exception as e:
                        print(f"Lỗi (attempt {attempt}) với {form_id} - đoạn {i+1}: {e}")
                        if attempt == MAX_RETRIES:
                            print("Bỏ qua sau khi retry quá nhiều lần.")
                            break
                        else:
                            sleep_time = BASE_DELAY * (2 ** (attempt - 1))
                            print(f"Đợi {sleep_time:.1f}s rồi thử lại...")
                            time.sleep(sleep_time)


In [1]:
!cp "/kaggle/input/dest-triples-26-10/dest_triples_25_10.json" /kaggle/working/


In [ ]:
import os
import json
import time
import re
import requests

# === DANH SÁCH CÁC API KEY ===
API_KEYS = []
current_key_index = 0
API_KEY = API_KEYS[current_key_index]

def update_api_key():
    """Chuyển sang API key kế tiếp khi bị lỗi 429 nhiều lần."""
    global current_key_index, API_KEY, API_URL
    current_key_index = (current_key_index + 1) % len(API_KEYS)
    API_KEY = API_KEYS[current_key_index]
    API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={API_KEY}"
    print(f"Đã chuyển sang API key thứ {current_key_index+1}/{len(API_KEYS)}")

# === Khởi tạo URL và headers ===
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={API_KEY}"
headers = {"Content-Type": "application/json"}

# === ĐƯỜNG DẪN INPUT/OUTPUT ===
# input_dir = "/kaggle/input/his-chunk-new-16-10/his_chunks"
output_path = "/kaggle/working/test_26_10_2.json"
processed_keys = set()

# === Đọc các đoạn đã xử lý (nếu có) ===
if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line.strip())
                key = (record["formId"], record["chunk_index"])
                processed_keys.add(key)
            except:
                continue
print(f"Đã tải {len(processed_keys)} đoạn đã xử lý trước đó.")

# === Lấy danh sách file JSON ===
# json_files = sorted([f for f in os.listdir(input_dir) if f.endswith(".json")])
# print(f"Tổng số file JSON cần xử lý: {len(json_files)}")

# === Tham số điều khiển ===
MAX_RETRIES = 5
BASE_DELAY = 2
WAIT_AFTER_N_REQUESTS = 10
LONG_SLEEP_SECONDS = 60

MAX_429_ERRORS_PER_KEY = 5  # sau 5 lần lỗi 429 sẽ đổi API key
consecutive_429_errors = 0
request_count = 0

MAX_400_ERRORS_PER_KEY = 5  # Sau 5 lỗi 400 liên tiếp thì đổi API key
consecutive_400_errors = 0

# === BẮT ĐẦU XỬ LÝ ===
# with open(output_path, "a", encoding="utf-8") as out_f:
#     for filename in json_files:
#         file_path = os.path.join(input_dir, filename)
out_f = open(output_path, "a", encoding="utf-8")

filename = "chunks_dest.json"
file_path = "/kaggle/input/new-chunk-25-10/chunks_dest.json"
try:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
except Exception as e:
    print(f"Lỗi đọc {file_path}: {e}")

print(f"\n=== Đang xử lý file: {file_path} ({len(data)} đoạn lớn) ===")

for i, form in enumerate(data):
    form_id = form.get("id", f"{filename}_unknown")
    chunk = form.get("text", "").strip()
    if not chunk:
        continue

    key = (form_id, 0)
    if key in processed_keys:
        print(f"Bỏ qua {form_id} (đã xử lý)")
        continue


    prompt = f"""Bạn là một công cụ trích xuất thông tin để xây dựng đồ thị tri thức du lịch – lịch sử Việt Nam.
Mục tiêu: Từ một đoạn văn bản đầu vào (tiếng Việt), hãy trích xuất các bộ ba (triple) theo định dạng JSON:
[{{"subject": "...", "subject_type": "...", "relation": "...", "object": "...", "object_type": "..."}}]
Yêu cầu:
- Mỗi dòng đại diện cho một quan hệ.
- Không nên dùng danh từ làm tên relation.
- Thực thể là danh từ riêng.
- Không nên dùng quan hệ quá dài hoặc mang nghĩa mơ hồ như "liên_quan_đến", "có_liên_hệ_với".
- Nếu loại thực thể là địa điểm chỉ địa lý hành chính như "tỉnh", "thành phố", "quận", "huyện", "thị xã", "xã", "phường", "thôn", "bản", "ấp" thì lưu loại thực thể là cấp hành chính đó luôn.
- Các loại thực thể có thể bao gồm:
Địa điểm (tỉnh, thành phố, quận, huyện, thị xã, xã, phường, thôn, bản, ấp)
Nhân vật lịch sử
Điểm du lịch (chùa, bãi biển, công viên, danh thắng)
Di sản (văn hóa, thiên nhiên, hỗn hợp)
Di tích
Cổ vật
Công trình lịch sử
Lễ hội (sự kiện văn hóa – du lịch định kỳ)
Tổ chức / Cơ quan quản lý (sở du lịch, trung tâm bảo tồn)
- Các loại quan hệ có thể bao gồm:
Nằm ở
Được công nhận là 
Tổ chức tại
Tưởng niệm
Quản lý bởi 
Lấy cảm hứng từ 
Thuộc loại hình 
Gắn với thời kỳ lịch sử
Gắn với lễ hội
Xảy ra vào
Gần địa điểm du lịch
- Ví dụ: “Lễ hội Đền Trần được tổ chức tại tỉnh Nam Định nhằm tưởng nhớ công lao của Trần Hưng Đạo.”
Kết quả: [
          {{
            "subject": "Lễ hội Đền Trần ",
            "subject_type": "Lễ_hội",
            "relation": "tổ_chức_tại",
            "object": "Nam Định",
            "object_type": "Tỉnh"
          }},
          {{
            "subject": "Lễ hội Đền Trần ",
            "subject_type": "Lễ_hội",
            "relation": "tưởng_niệm",
            "object": "Trần Hưng Đạo",
            "object_type": "Nhân_vật_lịch_sử"
          }}
        ]
Đoạn văn đầu vào:
{chunk}
→ Triple:
Hãy trả về chỉ các triple dưới dạng JSON như trên, không thêm lời giải thích."""

    payload = {"contents": [{"parts": [{"text": prompt.strip()}]}]}

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(API_URL, headers=headers, json=payload)
            if response.status_code == 429:
                consecutive_429_errors += 1
                raise Exception("429 Too Many Requests")

            elif response.status_code == 400:
                consecutive_400_errors += 1
                print(f"Lỗi 400 (Bad Request) tại {form_id} - đoạn {i+1}")
                print("Response content:", response.text[:300])
    
                # Nếu quá nhiều lỗi 400 → đổi key
                if consecutive_400_errors >= MAX_400_ERRORS_PER_KEY:
                    print(f"Quá {MAX_400_ERRORS_PER_KEY} lỗi 400 liên tiếp → đổi API key.")
                    update_api_key()
                    consecutive_400_errors = 0
                    time.sleep(10)
                    continue
    
                # Thử làm sạch prompt rồi retry 1 lần
                if attempt == 1:
                    cleaned_prompt = re.sub(r"[^\x00-\x7F]+", " ", prompt)
                    payload["contents"][0]["parts"][0]["text"] = cleaned_prompt
                    print("Đang thử lại với prompt đã làm sạch ký tự đặc biệt...")
                    time.sleep(3)
                    continue
                else:
                    print("Bỏ qua đoạn do lỗi 400 lặp lại.")
                    break

            response.raise_for_status()
            res_data = response.json()
            generated_text = res_data["candidates"][0]["content"]["parts"][0]["text"]

            # Reset lỗi 429 nếu thành công
            consecutive_429_errors = 0
            consecutive_400_errors = 0

            # Làm sạch JSON
            generated_text = re.sub(r"^```json\s*", "", generated_text.strip())
            generated_text = re.sub(r"\s*```$", "", generated_text.strip())

            try:
                parsed = json.loads(generated_text)
                if isinstance(parsed, list):
                    triples = parsed
                elif isinstance(parsed, dict) and "triples" in parsed:
                    triples = parsed["triples"]
                else:
                    raise ValueError("Format không hợp lệ.")
            except Exception as parse_err:
                print(f"Không parse được JSON: {parse_err}")
                print("Nội dung trả về:\n", generated_text)
                triples = []

            record = {
                "formId": form_id,
                "chunk_index": 0,
                "triples": triples,
                "text": chunk,
                "file": filename,
                "api_key_index": current_key_index
            }

            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_f.flush()

            print(f"{form_id} - đoạn {i+1}/{len(chunk)} ({len(triples)} triple)")

            # Delay
            request_count += 1
            if request_count % WAIT_AFTER_N_REQUESTS == 0:
                print(f"Nghỉ {LONG_SLEEP_SECONDS} giây sau {WAIT_AFTER_N_REQUESTS} request...")
                time.sleep(LONG_SLEEP_SECONDS)
            else:
                time.sleep(2)
            break

        except Exception as e:
            print(f"Lỗi (attempt {attempt}) với {form_id} - đoạn {i+1}: {e}")

            # Nếu bị lỗi 429 quá nhiều lần → đổi API key
            if consecutive_429_errors >= MAX_429_ERRORS_PER_KEY:
                update_api_key()
                consecutive_429_errors = 0
                print("Đang chuyển sang API key khác và tạm nghỉ 15 giây...")
                time.sleep(15)
                continue  # thử lại với key mới

            if attempt == MAX_RETRIES:
                print("Bỏ qua sau khi retry quá nhiều lần.")
                break
            else:
                sleep_time = BASE_DELAY * (2 ** (attempt - 1))
                print(f"Đợi {sleep_time:.1f}s rồi thử lại...")
                time.sleep(sleep_time)


Đã tải 45 đoạn đã xử lý trước đó.

=== Đang xử lý file: /kaggle/input/new-chunk-25-10/chunks_dest.json (1928 đoạn lớn) ===
Bỏ qua 1_2 (đã xử lý)
Bỏ qua 1_3 (đã xử lý)
Bỏ qua 1_4 (đã xử lý)
Bỏ qua 2_0 (đã xử lý)
Bỏ qua 2_1 (đã xử lý)
Bỏ qua 2_2 (đã xử lý)
Bỏ qua 2_3 (đã xử lý)
Bỏ qua 2_4 (đã xử lý)
Bỏ qua 3_0 (đã xử lý)
Bỏ qua 3_1 (đã xử lý)
Bỏ qua 3_2 (đã xử lý)
Bỏ qua 4_0 (đã xử lý)
Bỏ qua 4_1 (đã xử lý)
Bỏ qua 4_2 (đã xử lý)
Bỏ qua 4_3 (đã xử lý)
Bỏ qua 5_0 (đã xử lý)
Bỏ qua 5_1 (đã xử lý)
Bỏ qua 5_2 (đã xử lý)
Bỏ qua 6_0 (đã xử lý)
Bỏ qua 6_1 (đã xử lý)
Bỏ qua 6_2 (đã xử lý)
Bỏ qua 6_3 (đã xử lý)
Bỏ qua 6_4 (đã xử lý)
Bỏ qua 6_5 (đã xử lý)
Bỏ qua 6_6 (đã xử lý)
Bỏ qua 6_7 (đã xử lý)
Bỏ qua 7_0 (đã xử lý)
Bỏ qua 7_1 (đã xử lý)
Bỏ qua 8_0 (đã xử lý)
Bỏ qua 8_1 (đã xử lý)
Bỏ qua 8_2 (đã xử lý)
Bỏ qua 10_0 (đã xử lý)
Bỏ qua 10_1 (đã xử lý)
Bỏ qua 10_2 (đã xử lý)
Bỏ qua 10_3 (đã xử lý)
Bỏ qua 10_4 (đã xử lý)
Bỏ qua 10_5 (đã xử lý)
Bỏ qua 10_6 (đã xử lý)
Bỏ qua 10_7 (đã xử lý)
Bỏ qua 10_8

✅ Cập nhật hành chính: Tịnh Biên (huyện) → Châu Đốc - Tịnh Biên (thành_phố) [sáp_nhập]
✅ Cập nhật hành chính: Châu Đốc (thành_phố) → Châu Đốc - Tịnh Biên (thành_phố) [sáp_nhập]


In [3]:
# === BƯỚC CUỐI: NÉN FILE KẾT QUẢ ĐỂ TẢI VỀ ===
import shutil

zip_name = "/kaggle/worki0ng/hist_triples_v3.zip"
output_file = "/kaggle/working/history_triples_v3.json"

if os.path.exists(output_file):
    print("📦 Đang nén file kết quả để tải về...")
    shutil.make_archive("/kaggle/working/history_triples_v3", 'zip', "/kaggle/working", "history_triples_v3.json")
    print(f"✅ Đã tạo file ZIP: {zip_name}")
    print("\n👉 Bạn có thể tải file này ở phần Output Files (bên phải hoặc tab 'Output') trên Kaggle Notebook.")
else:
    print("⚠️ Không tìm thấy file kết quả để nén.")


⚠️ Không tìm thấy file kết quả để nén.


In [ ]:
from IPython.display import FileLink
FileLink("/kaggle/working/history_triples_v3.zip")


In [ ]:
from IPython.display import FileLink
FileLink("/kaggle/working/history_triples_v3 (1) (3) (2).json")